# 04 — Premium calculations

Derives and sanity-checks the formulas in [METHODOLOGY.md §3](../docs/methodology.md) before they are served by the backend. The backend keeps its own copy of these functions (`backend/app/actuarial.py`), and `backend/tests/test_actuarial.py` asserts the two agree, so what this notebook reviews is what the API serves.

Notation: `v = 1/(1+i)`, `k|qx` = probability a life aged x dies in year k+1, `äx` = life annuity-due.

In [1]:
import numpy as np
import pandas as pd

from vitaemx_research import conapo, premiums

PROCESSED = conapo.RAW_DIR.parent / "processed"
life_tables = pd.read_csv(PROCESSED / "life_tables.csv")
fitted = pd.read_csv(PROCESSED / "fitted_qx.csv")

## 1. The survivor series the premiums use

Premiums use the *fitted* `qx` (smoothed for ages 30–90, raw outside), rebuilt into an `lx` series from a radix of 100,000. That is the same construction the backend does at cold start.

In [2]:
def lx_from_qx(qx: np.ndarray, radix: float = 100_000) -> np.ndarray:
    lx = np.empty(len(qx))
    lx[0] = radix
    for i in range(1, len(qx)):
        lx[i] = lx[i - 1] * (1 - qx[i - 1])
    return lx

nat = fitted[(fitted.state_code == 0) & (fitted.sex == "total")].sort_values("age")
lx = lx_from_qx(nat.qx_fitted.to_numpy())
ages = nat.age.to_numpy()

## 2. Worked example: age 35, 20-year term, i = 5%

In [3]:
i = 0.05
x = 35
n = 20
idx = int(np.flatnonzero(ages == x)[0])

A_term = premiums.term_insurance_nsp(lx, idx, n, i)
a_term = premiums.annuity_due(lx, idx, n, i)
A_whole = premiums.whole_life_nsp(lx, idx, i)
a_whole = premiums.annuity_due(lx, idx, None, i)

pd.DataFrame({
    "product": ["term 20", "whole life"],
    "net single premium (per 1 of benefit)": [A_term, A_whole],
    "annuity-due factor": [a_term, a_whole],
    "level annual premium (per 1 of benefit)": [A_term / a_term, A_whole / a_whole],
}).round(6)

,product,net single premium (per 1 of benefit),annuity-due factor,level annual premium (per 1 of benefit)
0,term 20,0.047458,12.728502,0.003728
1,whole life,0.154305,17.759590,0.008689


## 3. Checks that must hold

- Whole-life NSP at 0% interest is exactly 1 (a certain payment of 1, undiscounted).
- Term NSP grows with the term and with age, and falls as interest rises.
- Whole-life equals term insurance to the end of the table.

In [4]:
assert abs(premiums.whole_life_nsp(lx, idx, 0.0) - 1.0) < 1e-9
assert premiums.term_insurance_nsp(lx, idx, 10, i) < premiums.term_insurance_nsp(lx, idx, 20, i)
assert premiums.term_insurance_nsp(lx, idx, 20, i) < premiums.term_insurance_nsp(lx, idx + 10, 20, i)
assert premiums.term_insurance_nsp(lx, idx, 20, 0.08) < premiums.term_insurance_nsp(lx, idx, 20, 0.03)
assert abs(premiums.whole_life_nsp(lx, idx, i) - premiums.term_insurance_nsp(lx, idx, len(lx) - idx, i)) < 1e-12

## 4. Premium schedule by age

Annual level premium per 1,000 of sum assured, 20-year term, 5%, national both-sexes table.

In [5]:
rows = []
for age in range(20, 71, 5):
    k = int(np.flatnonzero(ages == age)[0])
    A = premiums.term_insurance_nsp(lx, k, 20, i)
    a = premiums.annuity_due(lx, k, 20, i)
    rows.append({"age": age, "annual premium per 1000": 1000 * A / a})
pd.DataFrame(rows).round(2)

,age,annual premium per 1000
0,20,2.31
1,25,2.76
2,30,3.14
3,35,3.73
4,40,4.68
5,45,6.21
6,50,8.66
7,55,12.54
8,60,18.58
9,65,27.76
